# Zadoff–Chu dual-path TX → DAC → ADC → BRAM loopback test

This notebook verifies both TX loading paths with the same waveform:

```text
Path A: PYNQ → AXI-Lite command 0 → TX BRAM
Path B: PYNQ/CMA → DDR → S_AXI_HP0_FPD → command 3 → TX BRAM
Both:  TX BRAM → wave_player → RFDC DAC → attenuated cable → RFDC ADC
       → wave_capture → RX BRAM → iq_bram_bridge_rx → PYNQ
```

Each path writes and verifies the complete 256 × 128-bit (4096-byte) frame, runs an RF loopback capture, and plots the same noise-referenced TX/RX spectrum used by the earlier test. Path B is loaded only after Path A playback has completed, so the DDR result cannot be masked by the direct write.

> **Safety:** do not set `ATTENUATION_CONFIRMED=True` unless the physical DAC-to-ADC path includes suitable attenuation. Start with the low digital amplitude below.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np

PROJECT_DIR = Path.cwd()
BIT_NAME = 'rfsoc_waveform_tx_rx.bit'

ATTENUATION_CONFIRMED = False  # Change to True only after checking the RF path.
AMPLITUDE = 0.10              # Begin below the earlier 0.9 setting.
CENTER_MHZ = 1000.0
DURATION_S = 0.040

if not ATTENUATION_CONFIRMED:
    raise RuntimeError(
        'Confirm suitable DAC-to-ADC attenuation before continuing.'
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from rfsoc_sounder_ddr import RFSoCSounder
from zadoff_chu_bram_signal import ZadoffChuBramSignal

## 1. Program the overlay and run startup checks

The upgraded self-test expects `ddr_bram_bridge_tx` at `0xA0000000`; the RX bridge remains unchanged.

In [ ]:
sounder = RFSoCSounder(
    project_dir=PROJECT_DIR,
    bit_name=BIT_NAME,
)

startup_report = sounder.startup(
    configure_clocks=True,
    lmk_freq_mhz=245.76,
    lmx_freq_mhz=491.52,
    download=True,
    run_self_test=True,
    capture_smoke_test=True,
)

sounder.print_self_test(startup_report)
print()
print(sounder.summary())

assert startup_report['pass']
assert 'ddr_bram_bridge_tx' in sounder.overlay.ip_dict

## 2. Configure the DAC and ADC Fine NCOs

These settings match the earlier verified 1 GHz loopback test.

In [ ]:
tx_settings = sounder.configure_tx(
    center_mhz=CENTER_MHZ,
    nyquist_zone=1,
    duration_s=max(DURATION_S, 0.05),
    phase_offset_deg=0.0,
)

rx_settings = sounder.configure_rx(
    center_mhz=CENTER_MHZ,
    nyquist_zone=1,
    calibration_mode=2,
    dither=True,
    capture_timeout_s=0.5,
    settle_s=0.010,
)

print('TX settings:', tx_settings)
print('RX settings:', rx_settings)

## 3. Generate and pack the 1024-sample Zadoff–Chu frame

Each `uint32` sample contains `{Q[15:0], I[15:0]}`. Four consecutive samples form one 128-bit DDR/BRAM word.

In [ ]:
signal = ZadoffChuBramSignal(
    n_samples=1024,
    sc_range=(-250, 250),
    root=3,
    zc_q=0.0,
    amplitude=AMPLITUDE,
)

assert signal.verify_packing()
assert signal.bram_data.dtype == np.uint32
assert signal.bram_data.shape == (1024,)
assert signal.bram_data.nbytes == 4096

active_unshifted = np.ones(1024, dtype=bool)
active_unshifted[
    signal.sc_range[1]:1024 + signal.sc_range[0]
] = False
active_mask = np.fft.fftshift(active_unshifted)

waveform = sounder.set_waveform(
    packed=signal.get_bram_data(copy=True),
    active_mask=active_mask,
)

signal.print_summary()
print('I range:', int(waveform.i.min()), 'to', int(waveform.i.max()))
print('Q range:', int(waveform.q.min()), 'to', int(waveform.q.max()))

## 4A. Original PYNQ → AXI-Lite → TX BRAM path

This baseline uses bridge command 0 for 256 single-word writes, followed by a complete BRAM readback.

In [ ]:
direct_load_report = sounder.load_waveform_direct(verify=True)

print('Transport:', direct_load_report['transport'])
print('Bytes written:', direct_load_report['bytes_copied'])
print('128-bit words written:', direct_load_report['words_written'])
print('Packed mismatches:', direct_load_report['packed_mismatches'])
print('I mismatches:', direct_load_report['i_mismatches'])
print('Q mismatches:', direct_load_report['q_mismatches'])

assert direct_load_report['bytes_copied'] == 4096
assert direct_load_report['words_written'] == 256
assert direct_load_report['packed_mismatches'] == 0
assert direct_load_report['i_mismatches'] == 0
assert direct_load_report['q_mismatches'] == 0

## 4B. RF loopback and spectrum using the direct BRAM path

In [ ]:
capture_direct = sounder.transmit_and_capture(
    duration_s=DURATION_S,
    settle_s=0.010,
    capture_timeout_s=0.5,
    auto_load_waveform=False,
    require_frame_count_match=True,
)

stats_direct = sounder.capture_statistics(capture_direct)
print('Direct captured words:', capture_direct.captured_words)
print('Direct frames completed:', capture_direct.frames_completed)
print('Direct capture statistics:', stats_direct)

assert capture_direct.captured_words == 256
assert capture_direct.frames_completed == capture_direct.frames_requested
assert stats_direct['nonzero_i'] > 0
assert stats_direct['nonzero_q'] > 0

spectrum_direct = sounder.plot_spectra(
    capture_direct,
    rf_axis=True,
    noise_referenced=True,
    floor_db=-30.0,
)

print(
    'Direct RX mean active/noise:',
    spectrum_direct['rx_mean_active_above_noise_db'],
    'dB',
)

## 5A. DDR → HP0 → TX BRAM path

`load_waveform_ddr()` allocates and flushes a contiguous CMA buffer, executes bridge command 3, then reads back all 1024 packed IQ samples. This overwrites the complete frame left by Path A and does not use command 0 to write TX BRAM.

In [ ]:
load_report = sounder.load_waveform_ddr(verify=True)

print('Transport:', load_report['transport'])
print('DDR physical address:', hex(load_report['physical_address']))
print('Bytes copied:', load_report['bytes_copied'])
print('128-bit words copied:', load_report['words_written'])
print('Packed mismatches:', load_report['packed_mismatches'])
print('I mismatches:', load_report['i_mismatches'])
print('Q mismatches:', load_report['q_mismatches'])

assert load_report['bytes_copied'] == 4096
assert load_report['words_written'] == 256
assert load_report['packed_mismatches'] == 0
assert load_report['i_mismatches'] == 0
assert load_report['q_mismatches'] == 0

## 5B. Compare TX BRAM write time

Readback verification is excluded from all write-time numbers. `DDR first-load` includes the one-time CMA allocation; `DDR warm-load` assumes that the same allocated buffer is reused.

In [ ]:
payload_bytes = 4096
direct_write_s = direct_load_report['write_time_s']
ddr_allocate_s = load_report['allocation_time_s']
ddr_prepare_s = load_report['buffer_prepare_time_s']
ddr_hardware_s = load_report['write_time_s']
ddr_warm_load_s = ddr_prepare_s + ddr_hardware_s
ddr_first_load_s = load_report['load_time_s']

def throughput_mb_s(elapsed_s):
    return payload_bytes / elapsed_s / 1e6

timing_comparison = {
    'direct_write_s': direct_write_s,
    'ddr_allocation_s': ddr_allocate_s,
    'ddr_buffer_prepare_s': ddr_prepare_s,
    'ddr_hardware_copy_s': ddr_hardware_s,
    'ddr_warm_load_s': ddr_warm_load_s,
    'ddr_first_load_s': ddr_first_load_s,
    'hardware_copy_speedup': direct_write_s / ddr_hardware_s,
    'warm_load_speedup': direct_write_s / ddr_warm_load_s,
    'first_load_speedup': direct_write_s / ddr_first_load_s,
}

print(f"PS direct AXI-Lite write : {direct_write_s * 1e3:10.3f} ms  "
      f"{throughput_mb_s(direct_write_s):10.3f} MB/s")
print(f"DDR CMA allocation       : {ddr_allocate_s * 1e3:10.3f} ms")
print(f"DDR fill + cache flush   : {ddr_prepare_s * 1e3:10.3f} ms")
print(f"DDR hardware BRAM copy   : {ddr_hardware_s * 1e3:10.3f} ms  "
      f"{throughput_mb_s(ddr_hardware_s):10.3f} MB/s")
print(f"DDR warm total load      : {ddr_warm_load_s * 1e3:10.3f} ms  "
      f"{throughput_mb_s(ddr_warm_load_s):10.3f} MB/s")
print(f"DDR first total load     : {ddr_first_load_s * 1e3:10.3f} ms  "
      f"{throughput_mb_s(ddr_first_load_s):10.3f} MB/s")
print(f"Hardware-copy speedup    : {timing_comparison['hardware_copy_speedup']:.2f}x")
print(f"Warm-load speedup        : {timing_comparison['warm_load_speedup']:.2f}x")
print(f"First-load speedup       : {timing_comparison['first_load_speedup']:.2f}x")

assert direct_write_s > 0
assert ddr_hardware_s > 0
assert ddr_first_load_s >= ddr_hardware_s

## 5C. Play the DDR-loaded frame and capture one ADC frame

`auto_load_waveform=False` is intentional: it guarantees that nothing rewrites TX BRAM after the verified DDR transfer.

In [ ]:
capture = sounder.transmit_and_capture(
    duration_s=DURATION_S,
    settle_s=0.010,
    capture_timeout_s=0.5,
    auto_load_waveform=False,
    require_frame_count_match=True,
)

print('Captured words:', capture.captured_words)
print('Frames requested:', capture.frames_requested)
print('Frames completed:', capture.frames_completed)
print('Capture time:', capture.capture_time_s, 's')
print('Total time:', capture.total_time_s, 's')

assert capture.captured_words == 256
assert capture.frames_completed == capture.frames_requested

## 6. Inspect the DDR-path ADC statistics before interpreting the spectrum

In [ ]:
stats = sounder.capture_statistics(capture)

for key, value in stats.items():
    print(f'{key:24s}: {value}')

assert stats['nonzero_i'] > 0, 'ADC I capture is zero'
assert stats['nonzero_q'] > 0, 'ADC Q capture is zero'

if stats['peak_full_scale_ratio'] >= 0.95:
    print('WARNING: ADC is close to clipping; increase attenuation.')

## 7. Plot the DDR-path TX/RX RF spectrum

The inactive-bin mean power is the 0 dB reference, matching the earlier notebook.

In [ ]:
spectrum = sounder.plot_spectra(
    capture,
    rf_axis=True,
    noise_referenced=True,
    floor_db=-30.0,
)

print(
    'TX mean active/noise:',
    spectrum['tx_mean_active_above_noise_db'],
    'dB',
)
print(
    'RX mean active/noise:',
    spectrum['rx_mean_active_above_noise_db'],
    'dB',
)

## 8. Compare direct-path and DDR-path RX spectra

In [ ]:
import matplotlib.pyplot as plt

direct_magnitude = np.abs(spectrum_direct['rx_fft'])
ddr_magnitude = np.abs(spectrum['rx_fft'])
direct_rx_db = 20.0 * np.log10(
    np.maximum(direct_magnitude / direct_magnitude.max(), 1e-15)
)
ddr_rx_db = 20.0 * np.log10(
    np.maximum(ddr_magnitude / ddr_magnitude.max(), 1e-15)
)
frequency_mhz = CENTER_MHZ + spectrum['frequency_mhz']
active = spectrum['active_mask']
active_delta_db = ddr_rx_db[active] - direct_rx_db[active]

print('Active-bin median DDR-direct delta:', np.median(active_delta_db), 'dB')
print(
    'Active-bin RMS DDR-direct delta:',
    np.sqrt(np.mean(active_delta_db ** 2)),
    'dB',
)

plt.figure(figsize=(13, 5))
plt.plot(frequency_mhz, direct_rx_db, label='Direct AXI-Lite → BRAM RX')
plt.plot(frequency_mhz, ddr_rx_db, label='DDR → BRAM RX', alpha=0.8)
plt.xlabel('Nominal RF frequency (MHz)')
plt.ylabel('RX magnitude relative to peak (dB)')
plt.title('Direct BRAM path vs DDR path')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 9. DDR-path circular correlation check

A repeated frame can begin at any sample phase, so circular correlation is used to compare TX and RX.

In [ ]:
tx_iq = waveform.iq.astype(np.complex128)
rx_iq = capture.iq.astype(np.complex128)

circular_correlation = np.fft.ifft(
    np.fft.fft(rx_iq) * np.conj(np.fft.fft(tx_iq))
)
peak_index = int(np.argmax(np.abs(circular_correlation)))
normalized_peak = float(
    np.abs(circular_correlation[peak_index])
    / np.sqrt(
        np.sum(np.abs(tx_iq) ** 2)
        * np.sum(np.abs(rx_iq) ** 2)
    )
)

print('Circular correlation peak index:', peak_index)
print('Normalized correlation peak:', normalized_peak)

## Optional: plot the channel magnitude and phase

In [ ]:
channel = sounder.plot_channel(
    capture,
    definition='standard',
    rf_axis=True,
)

print(
    'Estimated channel delay:',
    channel['estimated_delay_s'] * 1e9,
    'ns',
)

## Optional cleanup

Release the CMA buffer when no further waveform reloads are needed. TX playback no longer accesses DDR after the bridge copy completes.

In [ ]:
sounder.release_ddr_buffer()
print('DDR CMA buffer released')